This notebook processes the experimentally validated TSS sites. The raw table are the selected columns from table S2 of Chen *et al.* 2013. The deliverables of this notebook:

- Clean the raw table `data/raw/baculovirus/Baculovirus_TSS.csv`
- Extract positive and negative sets for TSS prediction task.
- Generate local datasets loadable as `datasets.DatasetDict` following [Hugging Face document](https://huggingface.co/docs/datasets/create_dataset). The processed datasets is the baculovirus counterpart of dataset for promoter existance prediction task in [`"InstaDeepAI/nucleotide_transformer_downstream_tasks_revised"`](https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised). The the datasets structures should be identical.

# Clean TSS Table

In [1]:
import pandas as pd

In [2]:
raw_df = pd.read_csv("../data/raw/baculovirus/Baculovirus_TSS.csv").dropna(axis=0, subset=['TSS'])

In [3]:
raw_df.shape

(216, 9)

Different from the authors' claim of 220 TSS, there are 4 sites not recorded in column "All TSS sites" in the original table. Since the original table was clearly manually curated, I decided not to challenge the reason of their removal and accept the authors' decision.

### Fill Missing

This table was designed for human read, thus some units are left blank intentedly.

In [4]:
ffill_cols = ["Direction", "ORF", "ORF start", "ORF end"]
raw_df[ffill_cols] = raw_df[ffill_cols].ffill()

In [5]:
raw_df.columns

Index(['TSS', 'Direction', 'TATAA_Distance_from_TSS', 'CAGT_Distance_from_TSS',
       'TAAG_Distance_from_TSS', 'ORF', 'ORF start', 'ORF end',
       'Early / Late'],
      dtype='str')

In [6]:
intcols = [
    'TSS', 
    'ORF start', 
    'ORF end'
]
raw_df[intcols] = raw_df[intcols].astype('int', errors='ignore')

In [7]:
raw_df.describe()

,TSS,TATAA_Distance_from_TSS,ORF start,ORF end
count,216.00000,75.000000,216.000000,216.000000
mean,66456.99537,-11.706667,65946.421296,66814.504630
std,38286.42905,23.959861,38290.847026,38332.276286
min,482.00000,-47.000000,503.000000,1009.000000
25%,30923.25000,-30.000000,30288.750000,30860.750000
50%,64537.00000,-15.000000,63678.000000,64374.500000
75%,99781.00000,-3.000000,99830.500000,100029.750000
max,133588.00000,44.000000,133591.000000,133836.000000


### Clean Motif Distance

For comma seperated "Distance_from_TSS" columns, choose the most distant value.

In [8]:
def pick_distance(s):
    if s:
        if isinstance(s, str):
            vals = [int(v) for v in s.split(",")]
            return min(vals)
        else:
            return s

In [9]:
distance_cols = [
    'TATAA_Distance_from_TSS', 
    'CAGT_Distance_from_TSS', 
    'TAAG_Distance_from_TSS', 
]
raw_df[distance_cols] = raw_df[distance_cols].map(pick_distance)

In [10]:
raw_df.TATAA_Distance_from_TSS.unique()

array([ nan,  43.,  -3., -30.,  -9., -29., -28., -32.,  33., -24., -15.,
       -25., -46., -44., -40.,  24.,  21., -27.,  44.,  -7.,  22.,   3.,
       -18., -33.,  23., -35.,  37., -26., -47.,  13., -31.,   6.])

In [11]:
raw_df.isna().sum()

TSS                          0
Direction                    0
TATAA_Distance_from_TSS    141
CAGT_Distance_from_TSS     151
TAAG_Distance_from_TSS      75
ORF                          0
ORF start                    0
ORF end                      0
Early / Late                 0
dtype: int64

In [12]:
raw_df.dtypes

TSS                          int64
Direction                      str
TATAA_Distance_from_TSS    float64
CAGT_Distance_from_TSS     float64
TAAG_Distance_from_TSS     float64
ORF                            str
ORF start                    int64
ORF end                      int64
Early / Late                   str
dtype: object

### Remove TSS inside ORF

In [13]:
tss_within_orf_idx = (((raw_df.TSS - raw_df["ORF start"]) * (raw_df.TSS - raw_df["ORF end"])) < 0)
tss_within_orf_idx.value_counts()

False    212
True       4
Name: count, dtype: int64

In [14]:
raw_df = raw_df[~tss_within_orf_idx].reset_index(drop=True)

In [15]:
raw_df.head()

,TSS,Direction,TATAA_Distance_from_TSS,CAGT_Distance_from_TSS,TAAG_Distance_from_TSS,ORF,ORF start,ORF end,Early / Late
0,482,+,NaN,NaN,-1.0,ptpase,503,1009,L
1,2272,-,NaN,NaN,-1.0,Ac-bro,1041,2027,L
2,2272,-,NaN,NaN,-1.0,ctx,2084,2245,L
3,2835,-,NaN,NaN,NaN,orf4,2295,2750,E
4,2738,+,NaN,NaN,-1.0,orf5,2779,3108,L


In [16]:
raw_df.to_csv('../data/interim/tss.csv', index=False)

# Extract Positive and Negative Sets

In [17]:
from Bio import SeqIO

In [18]:
# only 1 gb record in the file
with open("../data/raw/baculovirus/Baculoviridae/NC_001623.1.gb", 'r') as f:
    seq = next(iter(SeqIO.parse(f, format='gb')))

In [19]:
seq

SeqRecord(seq=Seq('GAATTCTACCCGTAAAGCGAGTTTAGTTTTGAAAAACAAATGACATCATTTGTA...GTA'), id='NC_001623.1', name='NC_001623', description='Autographa californica nucleopolyhedrovirus, complete genome', dbxrefs=['BioProject:PRJNA485481'])

In [20]:
len(seq)

133894

The sequence loaded by Biopython is 0-based. 

### TSS Table

The positions in TSS table are 1-based. Also note that the ORFs in TSS table are 1-based, double closed regions.

The motif distance follows the strand, with positive numbers indicating downstream and negative numbers indicating upstream. For TSS on positive strand, the motif starts at *TSS + Motif_distance* (1-based, included). For TSS on negative strand, the motif starts at *TSS - Motif_distance* (1-based, included)

For a TSS on positive strand, the motif distance -1 means the motif starts at 1 bp upstream. For a TSS on negative strand, the motif distance 10 means the motif starts at 10 bp downstream. In both cases the motif indices are smaller than TSS, i.e. on the left.

Note that the **distances in TSS table are not always precise**, maybe because of genome version updates, ORC recognition, or errors introduced in manual curation.

Take row 11 (1-based) as an example. TSS: 6885, TAAG motif distance: 4, ORF: 6917-7735. The true TAAG motif distance should be -1.

In [21]:
seq[6883:6890]

SeqRecord(seq=Seq('TAAGATT'), id='NC_001623.1', name='NC_001623', description='Autographa californica nucleopolyhedrovirus, complete genome', dbxrefs=[])

In [22]:
# pos strand
tss = 79902
motif_dist = -48
st = tss+motif_dist-1 
seq[st:st+5]

SeqRecord(seq=Seq('CAGTA'), id='NC_001623.1', name='NC_001623', description='Autographa californica nucleopolyhedrovirus, complete genome', dbxrefs=[])

In [23]:
# neg strand
tss = 2272
motif_dist = -1
st = tss-motif_dist 
seq[st-5:st].reverse_complement()

SeqRecord(seq=Seq('TAAGA'), id='<unknown id>', name='<unknown name>', description='<unknown description>', dbxrefs=[])

### Strategy

In accordance with the human promoter existance prediction task, we choose 300bp as the sequence length.

The 216 experimentally verified TSS sites are our positive set. However, there are duplicated TSS sites, which means one TSS controls multiple ORFs. In this case we only take care of the nearest ORF because we want to inspect the boundaries.

For positive TSS record:

- Deduplicate by TSS, only retain the records having the closest ORF.
- Do not include sequences in ORF (the coding region would be too obvious).
- Include all existing motifs.
- Strategy: randomly pick a position between ORF and the most downstream motif, extend 300 bp upstream.

For negative TSS record:

- Use intergenic regions (CDS region can be loaded from `seq.features` but UTR regions are not included). For this task, I'm 
- Keep distance from TSS regions on both strands.

### Define Boundaries

Identify motif related boundaries and ORF boundaries for each TSS as interim data for capturing final boundaries.

In [24]:
def motif_boundary(row_dct):
    "1-based boundary of must-be-included motif containing regions"
    tss_pos = row_dct['TSS']
    strand = row_dct["Direction"]
    motif_dists = [
        int(v) for k,v in row_dct.items() 
        if k.endswith("Distance_from_TSS") 
        and not pd.isna(v)
    ]
    if motif_dists:
        upstream_dist = min(min(motif_dists), 0)
        downstream_dist = max(max(motif_dists), 0)
        # include the motif length
        if downstream_dist:
            downstream_dist += 5
        # calculate boundary
        if strand == "+":
            return (tss_pos + upstream_dist, tss_pos + downstream_dist)
        elif strand == "-":
            return (tss_pos - downstream_dist, tss_pos - upstream_dist)
    else:
        return (tss_pos, tss_pos)

In [25]:
new_df = (
    raw_df.reset_index()[["TSS", "Direction", "ORF start", "ORF end"]]
    .rename(columns={
        "Direction": "strand",
        "ORF start": "orf_st", 
        "ORF end": "orf_ed"
    })
)

In [26]:
new_df[['motif_st', 'motif_ed']] = (
    raw_df.reset_index()
    .apply(
        lambda row: motif_boundary(row.to_dict()), 
        axis=1, 
        result_type='expand'
    )
)

### Remove duplicated TSS

In [27]:
def deduplicate_tss(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a new DataFrame with unique TSS values.

    Selection rule within a TSS group:
      - if strand == '+': choose the row with the smallest (orf_st - TSS)
      - if strand == '-': choose the row with the smallest (TSS - orf_ed)
    Rows with unique TSS are kept as-is.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: 'TSS', 'strand', 'orf_st', 'orf_ed'.

    Returns
    -------
    pd.DataFrame
        Subset of df (rows preserved, order restored by original index) with unique TSS.
    """
    out = df.copy()

    # Compute the per-row score according to strand (vectorized)
    score = pd.Series(index=out.index, dtype="float64")
    is_plus = out["strand"] == "+"
    is_minus = out["strand"] == "-"

    score.loc[is_plus]  = (out.loc[is_plus,  "orf_st"] - out.loc[is_plus,  "TSS"]).astype("float64")
    score.loc[is_minus] = (out.loc[is_minus, "TSS"]    - out.loc[is_minus, "orf_ed"]).astype("float64")

    out["_score"] = score.values

    # For each TSS, pick the index of the row with the minimum score
    keep_idx = out.groupby("TSS")["_score"].idxmin()

    result = out.loc[keep_idx].drop(columns="_score").sort_index()
    return result

In [28]:
new_df = deduplicate_tss(new_df).reset_index(drop=True)
new_df.shape

(205, 6)

In [29]:
new_df.to_csv('../data/interim/tss_dedup.csv', index=False)

In [30]:
new_df

,TSS,strand,orf_st,orf_ed,motif_st,motif_ed
0,482,+,503,1009,481,482
1,2272,-,2084,2245,2272,2273
2,2835,-,2295,2750,2835,2835
3,2738,+,2779,3108,2737,2738
4,3031,+,3089,3721,3030,3031
...,...,...,...,...,...,...
200,132968,-,132109,132387,132916,132969
201,132481,+,132526,133491,132450,132488
202,133577,+,133591,133836,133576,133577
203,133588,+,133591,133836,133576,133588


### Pick Regions as Positive Set

In [31]:
import random
random.seed(260316)

In [32]:
def pick_range(row_dct, length=300):
    "1-based closed interval"
    tss_pos = row_dct['TSS']
    strand = row_dct["strand"]
    if strand == "+":
        st = row_dct["motif_ed"] if row_dct["motif_ed"] <= row_dct["orf_st"] else tss_pos
        ed = min(row_dct["orf_st"] + 1, st + length)
        region_ed = random.choice(range(st, ed))
        region_st = region_ed - length + 1
    elif strand == "-":
        ed = row_dct['motif_st'] if row_dct['motif_st'] >= row_dct['orf_ed'] else tss_pos
        st = max(row_dct['orf_ed'], ed - length)
        region_st = random.choice(range(st, ed))
        region_ed = region_st + length - 1
    return (region_st, region_ed)    

In [33]:
new_df[['range_st', 'range_ed']] = (
    new_df
    .apply(
        lambda row: pick_range(row.to_dict()), 
        axis=1, 
        result_type='expand'
    )
)

In [34]:
new_df

,TSS,strand,orf_st,orf_ed,motif_st,motif_ed,range_st,range_ed
0,482,+,503,1009,481,482,198,497
1,2272,-,2084,2245,2272,2273,2266,2565
2,2835,-,2295,2750,2835,2835,2800,3099
3,2738,+,2779,3108,2737,2738,2454,2753
4,3031,+,3089,3721,3030,3031,2743,3042
...,...,...,...,...,...,...,...,...
200,132968,-,132109,132387,132916,132969,132652,132951
201,132481,+,132526,133491,132450,132488,132223,132522
202,133577,+,133591,133836,133576,133577,133278,133577
203,133588,+,133591,133836,133576,133588,133292,133591


### Save

In [35]:
new_df.drop(columns=["motif_st", "motif_ed"]).to_csv("../data/interim/tss_seq_range.csv", index=False)

Note that out of the 205 rows, there are 64 TSS that are within the ORF of another gene on the same strand. They are listed in `data/interim/tss_in_orf_same_strand_violations.csv` and summarized in `data/interim/tss_in_orf_same_strand_summary.csv`.

This `tss_seq_range.csv` will serve as the guide to get sequences for our positive sets, thus I'll commit it to git. This notebook will not be rerun to overwrite the results.